In [1]:
# Install the required packages
# %pip install -q litellm langchain langchain-community langchain-openai python-dotenv

Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
agentic-ai-practice 0.1.0 requires openai>=3.13.0, but you have openai 2.54.0 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import warnings
import logging

warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

# Now import LiteLLM normally
from litellm import completion

In [4]:
import litellm
litellm.suppress_debug_info = True

In [5]:
import warnings
import logging

# Keep the recording clean — suppress noisy AWS-related warnings
warnings.filterwarnings("ignore")
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

In [9]:
# Load API keys from a .env file
# Create a .env file in the same folder with:
# OPENAI_API_KEY=sk-...
# ANTHROPIC_API_KEY=sk-ant-...
# GROQ_API_KEY=gsk_...

import os
from dotenv import load_dotenv
load_dotenv()

# Quick check
print("OpenAI key loaded:    ", "✅" if os.getenv("OPENAI_API_KEY") else "❌")
print("Anthropic key loaded: ", "✅" if os.getenv("ANTHROPIC_API_KEY") else "❌")
print("Groq key loaded:      ", "✅" if os.getenv("GROQ_API_KEY") else "❌")

OpenAI key loaded:     ✅
Anthropic key loaded:  ❌
Groq key loaded:       ✅


In [18]:
from litellm import completion
# Same code, different providers — just change the `model` string!

# i don't have credits in openAi now 
# Call OpenAI
# response_openai = completion(
#     model="gpt-4o-mini",
#     messages=[{"role": "user", "content": "Explain RAG in one sentence."}]
# )
# print("🔵 OpenAI:    ", response_openai.choices[0].message.content)

# Call Groq (super fast inference)
response = completion(
    model="groq/openai/gpt-oss-20b", 
    messages=[
       {"role": "user", "content": "hello from litellm"}
   ],
)
print("🟢 Groq:      ", response.choices[0].message.content)

🟢 Groq:       Hello! How can I help you today?


In [22]:
from litellm import completion

prompt = "Explain RAG in one sentence."

# Just a list of model strings — that's the only configuration
providers = [
    ("🔵 OpenAI",     "gpt-4o-mini"),
    ("🟢 Groq",       "groq/openai/gpt-oss-20b"),
    ("🟣 Anthropic",  "claude-3-5-haiku-20241022"),
    ("🟡 Gemini",     "gemini/gemini-1.5-flash"),
]
for label, model in providers:
    try:
        r = completion(model=model, messages=[{"role": "user", "content": prompt}])
        print(f"{label:<15}: {r.choices[0].message.content[:80]}")
    except Exception as e:
        print(f"{label:<15}: ❌ {type(e).__name__}")

🔵 OpenAI       : ❌ RateLimitError
🟢 Groq         : RAG (Retrieval‑Augmented Generation) is a language‑model framework that first fe
🟣 Anthropic    : ❌ BadRequestError
🟡 Gemini       : ❌ NotFoundError


In [23]:
from litellm import completion

# Define a fallback chain: try GPT first, then Claude, then Groq
response = completion(
    model="gemini/gemini-1.5-flash",
    messages=[{"role": "user", "content": "What is an LLM Gateway?"}],
    fallbacks=[
        "gpt-4o-mini",
        "groq/openai/gpt-oss-20b"
    ]
)

print("Response:", response.choices[0].message.content[:200], "...")
print("\nWhich model actually answered?", response.model)

20:29:52 - LiteLLM:ERROR: fallback_utils.py:75 - Fallback attempt failed for model gemini/gemini-1.5-flash: litellm.NotFoundError: GeminiException - {
  "error": {
    "code": 404,
    "message": "models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.",
    "status": "NOT_FOUND"
  }
}
Traceback (most recent call last):
  File "d:\Agentic-AI-Practice\.venv\Lib\site-packages\litellm\llms\vertex_ai\gemini\vertex_and_google_ai_studio_gemini.py", line 2981, in async_completion
    response: Final = await client.post(
                      ^^^^^^^^^^^^^^^^^^
  File "d:\Agentic-AI-Practice\.venv\Lib\site-packages\litellm\litellm_core_utils\logging_utils.py", line 344, in async_wrapper
    result: Final = await func(*args, **kwargs)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\Agentic-AI-Practice\.venv\Lib\site-packages\litellm\llms\custom_h

Response: ## What Is an LLM Gateway?

In a nutshell, an **LLM Gateway** is an infrastructure layer that sits between your applications (or users) and one or many large‑language‑model (LLM) services.  
Think of  ...

Which model actually answered? openai/gpt-oss-20b


In [42]:
from litellm import completion, completion_cost

response = completion(
    model="groq/openai/gpt-oss-20b",
    messages=[{"role": "user", "content": "Write a haiku about AI."}]
)

# Pass custom_llm_provider explicitly to fix the ModelNotMappedError
cost = completion_cost(completion_response=response, custom_llm_provider="groq")

print("Response:    ", response.choices[0].message.content)
print("\nInput tokens: ", response.usage.prompt_tokens)
print("Output tokens:", response.usage.completion_tokens)
print(f"Cost:         ${cost:.8f}")   

Response:     Silent soft code hums,  
Minds of silicon bloom bright,  
Dawn of thought's new light.

Input tokens:  78
Output tokens: 393
Cost:         $0.00012375


In [43]:
import litellm

# 🧹 Reset any callbacks/strategies left over from earlier cells
litellm.callbacks = []
litellm.success_callback = []
litellm.failure_callback = []
litellm._async_success_callback = []
litellm._async_failure_callback = []

# Also clear any router-strategy state
litellm.cache = None

print("✅ LiteLLM state reset — ready for clean caching demo")

✅ LiteLLM state reset — ready for clean caching demo


In [50]:
%pip install soundfile


  Using cached soundfile-0.14.0-py2.py3-none-win_amd64.whl.metadata (18 kB)
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   -------------------- ------------------- 0.5/1.0 MB 1.7 MB/s eta 0:00:01
   ---------------------------------------- 1.0/1.0 MB 2.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [54]:
import litellm
import time
from litellm import completion
from litellm.caching import Cache

# Enable in-memory caching (you can also use Redis in production)
litellm.cache = Cache(type="local")

prompt = "What does LLM stand for? Answer in one line."

# First call — actually hits OpenAI
start = time.time()
r1 = completion(
    model="groq/openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}],
    caching=True
)
t1 = time.time() - start
print(f"❄️  First call (API):   {t1:.2f}s — {r1.choices[0].message.content}")

# Second call — served from cache, near-instant
start = time.time()
r2 = completion(
    model="groq/openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}],
    caching=True
)
t2 = time.time() - start
print(f"⚡ Second call (cache): {t2:.4f}s — {r2.choices[0].message.content}")

print(f"\n🚀 Speedup: {t1/t2:.1f}x faster, and ZERO cost on the second call!")

❄️  First call (API):   0.70s — LLM stands for Large Language Model.
⚡ Second call (cache): 0.0010s — LLM stands for Large Language Model.

🚀 Speedup: 669.7x faster, and ZERO cost on the second call!


In [59]:
import os
from litellm import Router

model_list = [
    {
        "model_name": "fast-cheap",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-20b",
            "api_key": os.getenv("GROQ_API_KEY"),
        },
    },
    {
        "model_name": "smart-coding",
        "litellm_params": {
            "model": "groq/openai/gpt-oss-120b",
            "api_key": os.getenv("GROQ_API_KEY"),
        },
    },
    {
        "model_name": "balanced",
        "litellm_params": {
            "model": "openai/gpt-4o-mini",
            "api_key": os.getenv("OPENAI_API_KEY"),
        },
    },
]

router = Router(
    model_list=model_list,
    fallbacks=[
        {"smart-coding": ["balanced"]},          # if Groq 120b fails → try GPT-4o-mini
        {"fast-cheap": ["balanced"]},            # if Groq 20b fails → try GPT-4o-mini
        {"balanced": ["smart-coding"]},          # optional: cross-fallback
    ],
    num_retries=2,        # retries within the same group before falling back
    cooldown_time=60,     # seconds a failed deployment is excluded
)   

fast_response = router.completion(
    model="fast-cheap",
    messages=[{"role": "user", "content": "Summarize: AI is changing software."}],
)

code_response = router.completion(
    model="smart-coding",
    messages=[{"role": "user", "content": "Write a Python function to reverse a string."}],
)

print("⚡ Fast/cheap (Groq):", fast_response.choices[0].message.content[:150])
print("\n🧠 Smart/coding (Groq gpt-oss-120b):\n", code_response.choices[0].message.content[:300])

⚡ Fast/cheap (Groq): AI is reshaping software across the entire lifecycle:

* **Development** – Code generation tools, natural‑language interfaces, and AI‑assisted IDEs ac

🧠 Smart/coding (Groq gpt-oss-120b):
 Here’s a simple, self‑contained Python function that returns the reverse of a given string. It works for regular strings, Unicode characters, and even empty inputs.

```python
def reverse_string(s: str) -> str:
    """
    Return a new string that is the reverse of the input string `s`.

    Paramet
